### **PHASE 1: Project Setup & Initial Data Exploration**


> Project: E-commerece RFM Customer Segmentation

> Auther: Anand Prajapati

> Dataset: Online Retail Dataset (UCI / Kaggle)


In [123]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [124]:
# Search for the file

import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file == 'online_retail_raw.csv':
            print(os.path.join(root, file))

/content/drive/MyDrive/RFM_Customer_Segmentation/Data/online_retail_raw.csv


In [125]:
# import libraries

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

Libraries imported successfully!


In [126]:
# Read the CSV

df = pd.read_csv("/content/drive/MyDrive/RFM_Customer_Segmentation/Data/online_retail_raw.csv", encoding='Latin-1')

print("Data loaded successfully")

Data loaded successfully


In [127]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [128]:
df.shape

(541909, 8)

In [129]:
# df.columns

df.columns.tolist()

['InvoiceNo',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'UnitPrice',
 'CustomerID',
 'Country']

In [130]:
df.dtypes

,0
InvoiceNo,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,object
UnitPrice,float64
CustomerID,float64
Country,object


In [131]:
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [132]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

In [133]:
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct.round(2)
})

print(missing_df[missing_df['Missing Count'] > 0])

             Missing Count  Missing %
Description           1454       0.27
CustomerID          135080      24.93


In [134]:
# Finding Duplicate

print(f"Total Duplicate: {df.duplicated().sum()}")

Total Duplicate: 5268


In [135]:
print(f"Total Transactions: {df.shape[0]:,}")
print(f"Total Columns: {df.shape[1]}")
print(f"Date Range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")
print(f"Unique Customers: {df['CustomerID'].nunique():,}")
print(f"Countries Covered: {df['Country'].nunique():,}")

Total Transactions: 541,909
Total Columns: 8
Date Range: 1/10/2011 10:04 to 9/9/2011 9:52
Unique Customers: 4,372
Countries Covered: 38


### **PHASE 2: Data Clearning & Validation**

In [111]:
df_clean = df.copy()
print(f"Workibg Copy Created")

Workibg Copy Created


In [136]:
print(f"Shape: {df_clean.shape[0]:,} Rows -- {df_clean.shape[1]} Columns")

Shape: 392,692 Rows -- 8 Columns


In [137]:
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])

In [138]:
# Remove nulls first
df_clean = df_clean.dropna(subset=["CustomerID"])

# Convert properly - this kills the .0 forever
df_clean["CustomerID"] = df_clean["CustomerID"].astype(float).astype(int).astype(str)

print(df_clean.isnull().sum())

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


In [139]:
df_clean.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


In [140]:
df_clean = df_clean.dropna(subset=["Description"])

In [141]:
# Remove Duplicate Values

before = df_clean.shape[0]
df_clean = df_clean.drop_duplicates()
after = df_clean.shape[0]
removed = before - after

print("Duplicates Removed Successfully")
print(f"Rows before: {before:,} ")
print(f"Removed: {removed:,} ")
print(f"Rows After: {after:,}")

Duplicates Removed Successfully
Rows before: 392,692 
Removed: 0 
Rows After: 392,692


In [142]:
# Removing Returns (Negative Quantity)

before = df_clean.shape[0]

# Keeping only Positive Quantities
df_clean = df_clean[df_clean["Quantity"] > 0]

# ~ simply flips every True to False and every False to True (NOT operator)
# It says give me opposite values instead of picking up C later of rows
df_clean = df_clean[~df_clean["InvoiceNo"].astype(str).str.startswith("C")]

after = df_clean.shape[0]
removed = before - after

In [143]:
print(f"Returns and cancellations removed")
print(f"Rows before: {before:,}")
print(f"Rows removed: {removed:,}")
print(f"Rows after: {after:,}")

Returns and cancellations removed
Rows before: 392,692
Rows removed: 0
Rows after: 392,692


In [144]:
# Remove Invalid Price

before = df_clean.shape[0]
df_clean = df_clean[df_clean["UnitPrice"] > 0]
after = df_clean.shape[0]
removed = before - after

print("Invalid prices removed")
print(f"Rows before: {before:,}")
print(f"Rows removed: {removed:,}")
print(f"Rows after: {after:,}")

Invalid prices removed
Rows before: 392,692
Rows removed: 0
Rows after: 392,692


**Feature Engineering - Creating a TotalPrice column**

In [150]:
df_clean["TotalPrice"] = df_clean['Quantity'] * df_clean["UnitPrice"]

print("TotalPrice column cretaed")
print(f"Simple Values: {df_clean["TotalPrice"].head(5)}")

TotalPrice column cretaed
Simple Values: 0    15.30
1    20.34
2    22.00
3    20.34
4    20.34
Name: TotalPrice, dtype: float64


In [147]:
df.dtypes

,0
InvoiceNo,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,object
UnitPrice,float64
CustomerID,float64
Country,object


In [148]:
df_clean.isnull().sum()

,0
InvoiceNo,0
StockCode,0
Description,0
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,0
Country,0


In [151]:
# Validation: Final Data Quality Check

print("="*55)
print("Final Data Quality Reports")
print("="*55)

print(f"\nFinal Shape : {df_clean.shape[0]:,} Rows -- {df_clean.shape[1]} Columns")
print(f"Date Range: {df_clean['InvoiceDate'].min()} to {df_clean['InvoiceDate'].max()}")
print(f"Unique Customer: {df_clean['CustomerID'].nunique():,}")
print(f"Unique Products: {df_clean['StockCode'].nunique():,}")
print(f"Countires: {df_clean['Country'].nunique():,}")
print(f"Total Revenue: {df_clean["TotalPrice"].sum():,.2f}")



print("\n"+"="*55)
print(f"Missing Values After Cleaning")
print("="*55)
print(f"\n{df_clean.isnull().sum()}")



print("\n"+"="*55)
print(f"Negative Values Check")
print("="*55)

print(f"\nNegative Quantity : {(df_clean['Quantity'] < 0).sum()}")
print(f"Negative UnitPrice: {(df_clean['UnitPrice'] < 0).sum()}")
print(f"Negative TotalPrice : {(df_clean["TotalPrice"] < 0).sum()}")



print("\n"+"="*55)
print(f"Data Types")
print("="*55)
print(f"\n{df_clean.dtypes}")



Final Data Quality Reports

Final Shape : 392,692 Rows -- 9 Columns
Date Range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00
Unique Customer: 4,338
Unique Products: 3,665
Countires: 37
Total Revenue: 8,887,208.89

Missing Values After Cleaning

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
TotalPrice     0
dtype: int64

Negative Values Check

Negative Quantity : 0
Negative UnitPrice: 0
Negative TotalPrice : 0

Data Types

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID             object
Country                object
TotalPrice            float64
dtype: object


In [152]:
# Clean Data
df_clean.head(5)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [153]:
# Saving The Clean Data

df_clean.to_csv("/content/online_retail_clean.csv", index=False)
print(f"Clean Data saved to drive")

Clean Data saved to drive


In [154]:
from google.colab import files

df_clean.to_csv("online_retail_cleans.csv", index=False)
files.download("online_retail_cleans.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>